# 00 - Phase 0 GATE: checkpoint -> released-score reproduction

**HARD prerequisite.** If the forward pass on LTC's released checkpoint does NOT reproduce
LTC's released softmax scores, **STOP - do not extract embeddings**. This failure is silent
(extraction runs, Phase 1 emits numbers, all meaningless); this check is the only alarm.

Pre-registered criteria: `pcc/reports/phase0_checkpoint_gate.md` (G1 accuracy, G2 NN match,
G3 true-prob curve, G4 label multiset). The check is **permutation-invariant** because LTC's
loaders use `shuffle=True` -> released rows are in an unrecoverable order (see release_audit.md).

**Run order:** (A) zero-image artifact pre-check (seconds) -> report back; then (B) full
image-based gate after the dataset download. Default dataset: **Pl@ntNet-300K** (feasible).


## 1. GPU check


In [ ]:
import subprocess
o = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(o.stdout if o.returncode==0 else 'WARNING: no GPU - forward pass will be slow on CPU.')


## 2. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''                       # this repo git URL (or upload manually)
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'

DATASET    = 'plantnet'               # 'plantnet' (start here) | 'inaturalist'
MODEL_TYPE = 'best'                   # best | last-epoch | double-dip
SPLIT      = 'val'                    # released split to check against (val or cal)

# gdown file IDs (from the LTC repo scripts) ------------------------------
GID_MODELS = '1tS-M-4IYyCGMeIxxyrgx2-XCZgdvw18S'   # models.zip (all 6 ResNet-50s)
GID_SCORES = {'plantnet':'1k_PPQV3VJT44hz02CcnbqPstjQo70vGr',
              'inaturalist':'1W8R8Jj2bhS2PbR-3X9vEw-WkanbOk6mq'}

# dataset image root (must exist BEFORE step B) ---------------------------
# Pl@ntNet-300K: download from Zenodo record 5645731; expect {root}/images/{val,test}/...
# iNaturalist-2018: needs val2018.json + val images (see release_audit.md logistics)
DATA_ROOT  = f'{DRIVE_ROOT}/data/{DATASET}'
INAT_ANN   = f'{DATA_ROOT}/val2018.json'   # iNaturalist only

# gate params (defaults match phase0_checkpoint_gate.md - do NOT loosen silently)
SUBSAMPLE      = 3000                 # val images to forward-pass for the gate
NN_SUBSAMPLE   = 1000
TOL_ACC, TOL_NN_LINF, TOL_NN_MEDIAN, TOL_CURVE = 0.002, 1e-4, 1e-5, 1e-3
# Pl@ntNet: ImageFolder class convention may differ from LTC -> G2 unconfirmed.
# iNaturalist: category_id convention is exact -> G2 valid.
CHECK_NN   = (DATASET == 'inaturalist')
SEED = 42
# =======================================================================
print('DATASET =', DATASET, '| CHECK_NN(G2) =', CHECK_NN)


## 3. Mount Drive + repo + pinned env + seed + versions


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
REPO_ROOT = os.getcwd()
subprocess.run(['pip','install','-q','gdown','scipy','scikit-learn'], check=False)
os.environ['PYTHONPATH'] = REPO_ROOT + os.pathsep + os.environ.get('PYTHONPATH','')
os.environ['PYTHONUTF8'] = '1'
from pcc.utils.seed import set_seed
from pcc.utils.device import get_device, gpu_name
from pcc.utils.io import environment_stamp
set_seed(SEED); DEVICE = get_device()
print('GPU:', gpu_name()); print('env:', environment_stamp()['packages'])


## 4. Download released checkpoint + scores (idempotent, to Drive)


In [ ]:
import os, glob, subprocess
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints/ltc_models'
SCORES_DIR = f'{DRIVE_ROOT}/released_scores/{DATASET}'
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(SCORES_DIR, exist_ok=True)

def _has(patt): return len(glob.glob(patt, recursive=True))>0
if not _has(f'{CKPT_DIR}/**/*{DATASET}*model*.pth'):
    subprocess.run(['gdown',GID_MODELS,'-O',f'{CKPT_DIR}/models.zip'], check=True)
    subprocess.run(['unzip','-o',f'{CKPT_DIR}/models.zip','-d',CKPT_DIR], check=True)
if not _has(f'{SCORES_DIR}/**/*{DATASET}*_softmax.npy'):
    subprocess.run(['gdown',GID_SCORES[DATASET],'-O',f'{SCORES_DIR}/{DATASET}.zip'], check=True)
    subprocess.run(['unzip','-o',f'{SCORES_DIR}/{DATASET}.zip','-d',SCORES_DIR], check=True)

CKPT = sorted(glob.glob(f'{CKPT_DIR}/**/{MODEL_TYPE}-{DATASET}-model.pth', recursive=True))
SCF  = sorted(glob.glob(f'{SCORES_DIR}/**/{MODEL_TYPE}-{DATASET}-model_{SPLIT}_softmax.npy', recursive=True))
assert CKPT, f'checkpoint not found under {CKPT_DIR} (inspect the unzip layout)'
assert SCF,  f'released {SPLIT} softmax not found under {SCORES_DIR}'
CKPT_PATH, SCORES_FOLDER = CKPT[0], os.path.dirname(SCF[0])
print('checkpoint:', CKPT_PATH); print('scores dir:', SCORES_FOLDER)


## 5. PRE-CHECK A - zero images (run first, report back)
Fast alarms needing no dataset download: checkpoint loads, head dim == released #classes,
released scores internally sane (their own argmax accuracy high).


In [ ]:
import numpy as np
from pcc.data.ltc_datasets import load_released_scores, NUM_CLASSES
from pcc.extract.backbones import load_ltc_resnet50
from pcc.eval.score_repro import top1_accuracy, sha256_file

rel_softmax, rel_labels = load_released_scores(SCORES_FOLDER, DATASET, SPLIT, MODEL_TYPE)
C_rel = rel_softmax.shape[1]; acc_rel = top1_accuracy(rel_softmax, rel_labels)
print(f'released {SPLIT}: N={len(rel_labels)} C={C_rel} self-accuracy={acc_rel:.4f} '
      f'labels[min,max]=[{rel_labels.min()},{rel_labels.max()}]')
assert C_rel == NUM_CLASSES[DATASET], f'released C {C_rel} != expected {NUM_CLASSES[DATASET]}'

model = load_ltc_resnet50(CKPT_PATH, NUM_CLASSES[DATASET], DEVICE)
head = model.fc.out_features
print('checkpoint fc out_features =', head)
assert head == C_rel, f'HEAD DIM {head} != released C {C_rel} -> wrong checkpoint/arch (STOP)'
print('PRE-CHECK A OK: head matches released classes; released scores sane.')
print('ckpt sha256:', sha256_file(CKPT_PATH)[:16], '...')


### === PRE-CHECK A ENDS ===
For the **pre-check-only** run (both datasets): stop here and report the printed output
(released N/C/self-accuracy, head_out_features, sha256) before running Step B. Step B
downloads the image dataset; do not start it until the gate target is chosen.


## 6. Build deterministic val dataset (STEP B - needs images)
`shuffle=False` so our order is reproducible; released order is permuted (fine - gate is
permutation-invariant). Uses a deterministic subsample of size `SUBSAMPLE`.


In [ ]:
import torch, numpy as np
from torch.utils.data import DataLoader, Subset
from pcc.data.ltc_datasets import test_transform, plantnet_val, INaturalist2018Val

tfm = test_transform()
if DATASET == 'plantnet':
    ds = plantnet_val(DATA_ROOT, SPLIT, transform=tfm)
    print('plantnet class_to_idx sample:', dict(list(ds.class_to_idx.items())[:5]))
else:
    ds = INaturalist2018Val(f'{DATA_ROOT}/', INAT_ANN, transform=tfm)

rng = np.random.default_rng(SEED)
sub = rng.choice(len(ds), min(SUBSAMPLE, len(ds)), replace=False)
loader = DataLoader(Subset(ds, sub.tolist()), batch_size=64, shuffle=False, num_workers=2)
print(f'{DATASET} {SPLIT}: total={len(ds)} forward-pass subsample={len(sub)}')


## 7. Forward pass -> our scores (float64 logits -> scipy softmax, exactly like LTC)


In [ ]:
from pcc.extract.backbones import forward_logits_and_embeddings
from pcc.eval.score_repro import top1_accuracy
mine_softmax, mine_labels, _ = forward_logits_and_embeddings(
    model, loader, DEVICE, capture_embeddings=False)
print('our subsample accuracy:', round(top1_accuracy(mine_softmax, mine_labels),4),
      '| released self-accuracy:', round(acc_rel,4))


## 8. EVALUATE GATE + STOP on FAIL + write report/marker


In [ ]:
import time, json, os
from pcc.eval.score_repro import evaluate_gate, sha256_file
from pcc.utils.io import write_report

res = evaluate_gate(mine_softmax, mine_labels, rel_softmax, rel_labels,
                    nn_subsample=NN_SUBSAMPLE, seed=SEED, tol_acc=TOL_ACC,
                    tol_nn_linf=TOL_NN_LINF, tol_nn_median=TOL_NN_MEDIAN,
                    tol_curve=TOL_CURVE, check_nn=CHECK_NN)
print(json.dumps(res, indent=2))

checksums = {'checkpoint': sha256_file(CKPT_PATH)}
report = write_report('pcc/reports', f'00_verify_checkpoint_{DATASET}_{SPLIT}',
    hypothesis='LTC released resnet50 reproduces released softmax on '+DATASET+' '+SPLIT,
    pass_criteria='G1 |acc|<=0.002; G2 NN>=99%@1e-4 (iNat only); G3 curve<=1e-3; G4 label multiset',
    config=dict(dataset=DATASET, split=SPLIT, model_type=MODEL_TYPE, subsample=int(len(sub)),
                check_nn=CHECK_NN, tolerances=dict(acc=TOL_ACC, nn_linf=TOL_NN_LINF,
                nn_median=TOL_NN_MEDIAN, curve=TOL_CURVE)),
    seed=SEED, results={**res, 'checksums':checksums, 'released_self_accuracy':acc_rel},
    conclusion=res['verdict'], started_at=time.time())
print('report:', report)

GATE_MARKER = f'{DRIVE_ROOT}/gates/GATE_PASSED_{DATASET}_{SPLIT}.json'
os.makedirs(os.path.dirname(GATE_MARKER), exist_ok=True)
if res['verdict'] == 'PASS':
    with open(GATE_MARKER,'w') as f:
        json.dump({'dataset':DATASET,'split':SPLIT,'checksums':checksums,'results':res}, f, indent=2)
    print('GATE PASSED - marker written. Extraction (01) may proceed.')
else:
    if os.path.exists(GATE_MARKER): os.remove(GATE_MARKER)
    raise SystemExit('GATE FAILED - STOP. Do NOT extract. Report results above. '
                     'Likely cause: transform/normalize/checkpoint/class-index convention.')
